# Experiment 5 - Image Compression with a Dense Autoencoder

**Aim:** Build an autoencoder that compresses 28x28 MNIST digits through a
32-dimensional bottleneck and reconstructs them, then measure the
reconstruction error and compression ratio.

**Theory in brief:** An autoencoder is trained to copy its input to its output
through a narrow hidden layer. The encoder maps the 784-pixel image to a
low-dimensional latent vector; the decoder reconstructs the image from that
vector. Because all information must pass through the bottleneck, the network
learns a compact representation. Training minimizes mean squared error between
the input and its reconstruction. The images are both the input and the target.

**Dataset:** MNIST, downloaded at runtime from
`https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz`.
No local dataset is used.

**Expected result (original lab record):** test MSE near **0.0137** with the
784 -> 64 -> 32 -> 64 -> 784 architecture after 10 epochs.

**Runtime:** about 1-2 minutes on CPU.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.decomposition import PCA

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Download and prepare MNIST

The loader uses `tf.keras.utils.get_file` with the explicit MNIST URL, then
reads the `x_train`, `y_train`, `x_test`, and `y_test` arrays from the NPZ
archive. Pixels are scaled to `[0, 1]` and each image is flattened from
28x28 to 784 values.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_mnist():
    path = tf.keras.utils.get_file(
        fname="mnist.npz",
        origin=MNIST_URL,
        cache_subdir="datasets",
    )
    with np.load(path) as data:
        return (
            (np.array(data["x_train"]), np.array(data["y_train"])),
            (np.array(data["x_test"]), np.array(data["y_test"])),
        )


(X_train_raw, y_train), (X_test_raw, y_test) = load_mnist()

X_train = (X_train_raw.astype("float32") / 255.0).reshape(-1, 784)
X_test = (X_test_raw.astype("float32") / 255.0).reshape(-1, 784)

print(f"Training images: {X_train.shape}")
print(f"Test images    : {X_test.shape}")
print(f"Pixel range    : [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Training labels: {y_train.shape}, test labels: {y_test.shape}")

assert X_train.shape[1] == 28 * 28
assert 0.0 <= X_train.min() <= X_train.max() <= 1.0

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[index].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"label {int(y_train[index])}", fontsize=8)
    ax.axis("off")
fig.suptitle("Sample MNIST digits", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp05_mnist_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Build the autoencoder

The architecture is symmetric around the bottleneck:

- Encoder: 784 -> 64 ReLU -> **32 ReLU (latent code)**
- Decoder: 32 -> 64 ReLU -> 784 sigmoid

The sigmoid output matches the normalized pixel range. The loss is mean
squared error, so the training target is the input image itself.

In [ ]:
LATENT_DIM = 32


def build_autoencoder(latent_dim=LATENT_DIM, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    encoder = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(784,), name="image"),
            tf.keras.layers.Dense(64, activation="relu", name="encoder_dense_64"),
            tf.keras.layers.Dense(latent_dim, activation="relu", name="latent_code"),
        ],
        name="encoder",
    )
    decoder = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(latent_dim,), name="latent_input"),
            tf.keras.layers.Dense(64, activation="relu", name="decoder_dense_64"),
            tf.keras.layers.Dense(784, activation="sigmoid", name="reconstruction"),
        ],
        name="decoder",
    )
    autoencoder = tf.keras.Sequential([encoder, decoder], name="autoencoder")
    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
    )
    return autoencoder, encoder, decoder


autoencoder, encoder, decoder = build_autoencoder()
autoencoder.summary()

## 4. Training

The model is trained with the images as both input and target. A validation
split of 10% monitors reconstruction quality and early stopping restores the
best weights.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=0,
    )
]

start_time = time.perf_counter()
history = autoencoder.fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    verbose=2,
    callbacks=callbacks,
)
training_seconds = time.perf_counter() - start_time

print(f"Training finished in {training_seconds:.2f} seconds")
print(f"Best validation MSE: {min(history.history['val_loss']):.6f}")

## 5. Test reconstruction error

In [ ]:
test_mse = autoencoder.evaluate(X_test, X_test, verbose=0)
reconstructions = autoencoder.predict(X_test, verbose=0)

per_image_mse = np.mean((X_test - reconstructions) ** 2, axis=1)
compression_ratio_dimensions = 784 / LATENT_DIM
original_bytes_per_image = 784
latent_bytes_per_image = LATENT_DIM * 4

print(f"Test MSE (mean over all test images): {test_mse:.6f}")
print(f"Per-image MSE: mean={per_image_mse.mean():.6f}, "
      f"min={per_image_mse.min():.6f}, max={per_image_mse.max():.6f}")
print()
print(f"Dimensional compression ratio: 784 / {LATENT_DIM} = {compression_ratio_dimensions:.1f}x")
print(f"Raw uint8 image: {original_bytes_per_image} bytes")
print(f"Float32 latent : {latent_bytes_per_image} bytes")
print(f"Byte-level ratio before entropy coding: "
      f"{original_bytes_per_image / latent_bytes_per_image:.2f}x")

assert reconstructions.shape == X_test.shape

## 6. Visual reconstruction comparison

In [ ]:
N_EXAMPLES = 10
sample_indices = np.arange(N_EXAMPLES)

fig, axes = plt.subplots(2, N_EXAMPLES, figsize=(16, 4))
for column, index in enumerate(sample_indices):
    axes[0, column].imshow(X_test[index].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title("Original", fontsize=8)
    axes[0, column].axis("off")

    axes[1, column].imshow(reconstructions[index].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title(f"MSE {per_image_mse[index]:.4f}", fontsize=8)
    axes[1, column].axis("off")

fig.suptitle("Original digits and autoencoder reconstructions", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp05_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Use the encoder and decoder separately

Compression is just the encoder forward pass. Decompression is the decoder
forward pass. The composition should reproduce the autoencoder output, which
is checked below.

In [ ]:
latent_codes = encoder.predict(X_test[:N_EXAMPLES], verbose=0)
decoded_from_latents = decoder.predict(latent_codes, verbose=0)
direct_reconstructions = autoencoder.predict(X_test[:N_EXAMPLES], verbose=0)

print(f"Latent code shape: {latent_codes.shape}")
print(f"Decoded shape    : {decoded_from_latents.shape}")
print(f"Max absolute difference between encoder-decoder and autoencoder: "
      f"{np.max(np.abs(decoded_from_latents - direct_reconstructions)):.2e}")

assert np.allclose(decoded_from_latents, direct_reconstructions, atol=1e-6)

## 8. Visualize the latent space with PCA

The bottleneck has 32 dimensions, which cannot be plotted directly. PCA
projects the latent codes to two dimensions; colors are the true digit labels,
used only for visualization. If the autoencoder learned useful structure,
digits of the same class should cluster.

In [ ]:
N_VISUALIZE = 3000
latent_visual = encoder.predict(X_test[:N_VISUALIZE], verbose=0)
pca = PCA(n_components=2, random_state=SEED)
latent_2d = pca.fit_transform(latent_visual)

fig, ax = plt.subplots(figsize=(7, 5.5))
scatter = ax.scatter(
    latent_2d[:, 0],
    latent_2d[:, 1],
    c=y_test[:N_VISUALIZE],
    cmap="tab10",
    s=5,
    alpha=0.7,
)
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title("PCA projection of the 32-dimensional latent codes")
fig.colorbar(scatter, ax=ax, ticks=range(10), label="Digit")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp05_latent_pca.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Inference: compress and reconstruct one image

In [ ]:
def compress_and_reconstruct(image):
    image = np.asarray(image, dtype=np.float32)
    if image.shape != (28, 28):
        raise ValueError(f"Expected a 28x28 image, got {image.shape}")
    flat = image.reshape(1, 784)
    if flat.max() > 1.0:
        flat = flat / 255.0
    latent = encoder.predict(flat, verbose=0)
    reconstruction = decoder.predict(latent, verbose=0).reshape(28, 28)
    mse = float(np.mean((flat.reshape(28, 28) - reconstruction) ** 2))
    return latent[0], reconstruction, mse


sample_index = 7
latent, reconstruction, sample_mse = compress_and_reconstruct(X_test_raw[sample_index])

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(X_test_raw[sample_index], cmap="gray")
axes[0].set_title(f"Original (label {int(y_test[sample_index])})")
axes[0].axis("off")
axes[1].imshow(reconstruction, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"Reconstruction (MSE {sample_mse:.4f})")
axes[1].axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp05_single_reconstruction.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Latent vector shape: {latent.shape}")
print(f"First 8 latent values: {latent[:8].round(3).tolist()}")

## 10. Save models and metrics

In [ ]:
autoencoder_path = OUTPUT_DIR / "exp05_autoencoder.keras"
encoder_path = OUTPUT_DIR / "exp05_encoder.keras"
decoder_path = OUTPUT_DIR / "exp05_decoder.keras"

autoencoder.save(autoencoder_path)
encoder.save(encoder_path)
decoder.save(decoder_path)

metrics = pd.DataFrame(
    [
        {
            "Test MSE": float(test_mse),
            "Latent dimensions": LATENT_DIM,
            "Dimensional ratio": float(compression_ratio_dimensions),
            "Training seconds": float(training_seconds),
        }
    ]
)
metrics.to_csv(OUTPUT_DIR / "exp05_metrics.csv", index=False)
history_df = pd.DataFrame(history.history)
history_df.to_csv(OUTPUT_DIR / "exp05_history.csv", index=False)

print(f"Saved autoencoder: {autoencoder_path.resolve()}")
print(f"Saved encoder    : {encoder_path.resolve()}")
print(f"Saved decoder    : {decoder_path.resolve()}")
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

## 11. Conclusion

The dense autoencoder learns a 32-dimensional representation of MNIST and
reconstructs test digits with low mean squared error. The original lab sheet
records a test MSE near **0.0137**, which is consistent with the architecture
used here. The encoder-decoder check confirms that compression and
decompression are the two halves of the same trained network.

The 784/32 dimensional ratio is **24.5x**. If the latent code is stored as
float32, the raw byte-level ratio is only about **6.1x**; entropy coding or
quantization would be needed to realize the full dimensional gain in storage.
This distinction is important when describing autoencoder compression.

**Artifacts created at runtime (not committed):**

- `outputs/exp05_autoencoder.keras`
- `outputs/exp05_encoder.keras`
- `outputs/exp05_decoder.keras`
- `outputs/exp05_metrics.csv`
- `outputs/exp05_history.csv`
- `outputs/exp05_mnist_samples.png`
- `outputs/exp05_reconstructions.png`
- `outputs/exp05_latent_pca.png`
- `outputs/exp05_single_reconstruction.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full training run was not performed locally.